# U.S. Airline Operational Performance Analytics 2025
## 04 — Load Data to PostgreSQL

## 1. Install / Import Libraries

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("psycopg2") is None:
    print("Installing psycopg2-binary...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "psycopg2-binary",
        "-q"
    ])

import psycopg2
import gzip
import getpass
import time
from pathlib import Path

print("Libraries ready.")


Libraries ready.


## 2. Detect Project Folder

In [ ]:
current_path = Path.cwd().resolve()

if (current_path / "data" / "database").exists():
    project_root = current_path
elif (current_path.parent / "data" / "database").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError("Folder data/database tidak ditemukan.")

database_path = project_root / "data" / "database"

print(f"Project root: {project_root}")
print(f"Database files path: {database_path}")


## 3. Validate Export Files

In [3]:
dim_airline_file = database_path / "dim_airline.csv"
dim_airport_file = database_path / "dim_airport.csv"
dim_date_file = database_path / "dim_date.csv"

fact_files = sorted(
    database_path.glob("fact_flights_2025_*.csv.gz")
)

for file in [
    dim_airline_file,
    dim_airport_file,
    dim_date_file
]:
    if not file.exists():
        raise FileNotFoundError(f"File not found: {file}")

if len(fact_files) != 12:
    raise ValueError(
        f"Expected 12 fact files, found {len(fact_files)}."
    )

print("Dimension files found: 3")
print(f"Fact files found: {len(fact_files)}")
print("Input validation passed.")


Dimension files found: 3
Fact files found: 12
Input validation passed.


## 4. Database Configuration

In [4]:
db_config = {
    "host": "localhost",
    "port": "5432",
    "dbname": "airline_analytics_2025",
    "user": "postgres"
}

print(f"Host: {db_config['host']}")
print(f"Port: {db_config['port']}")
print(f"Database: {db_config['dbname']}")
print(f"User: {db_config['user']}")

db_password = getpass.getpass("PostgreSQL password: ")


Host: localhost
Port: 5432
Database: airline_analytics_2025
User: postgres


## 5. Test PostgreSQL Connection

In [5]:
conn = psycopg2.connect(
    host=db_config["host"],
    port=db_config["port"],
    dbname=db_config["dbname"],
    user=db_config["user"],
    password=db_password
)

conn.autocommit = False
cursor = conn.cursor()

cursor.execute("SELECT current_database(), current_user;")
database_name, current_user = cursor.fetchone()

print(f"Connected database: {database_name}")
print(f"Connected user: {current_user}")
print("PostgreSQL connection successful.")


Connected database: airline_analytics_2025
Connected user: postgres
PostgreSQL connection successful.


## 6. Clear Existing Data

In [6]:
cursor.execute(
    "TRUNCATE TABLE fact_flights, dim_airline, dim_airport, dim_date;"
)

conn.commit()

print("Existing table data cleared.")


Existing table data cleared.


## 7. Helper Function for COPY

In [7]:
def copy_csv_to_postgres(
    connection,
    table_name,
    columns,
    file_path,
    compressed=False
):
    local_cursor = connection.cursor()

    column_sql = ", ".join(columns)

    copy_sql = f"""
        COPY {table_name} ({column_sql})
        FROM STDIN
        WITH (
            FORMAT CSV,
            HEADER TRUE
        )
    """

    start_time = time.time()

    if compressed:
        with gzip.open(
            file_path,
            mode="rt",
            encoding="utf-8",
            newline=""
        ) as file:
            local_cursor.copy_expert(
                copy_sql,
                file
            )
    else:
        with open(
            file_path,
            mode="r",
            encoding="utf-8",
            newline=""
        ) as file:
            local_cursor.copy_expert(
                copy_sql,
                file
            )

    connection.commit()
    local_cursor.close()

    elapsed_time = time.time() - start_time

    print(
        f"Loaded {table_name} from {file_path.name} "
        f"in {elapsed_time:.1f} seconds."
    )


## 8. Load Dimension Tables

In [8]:
copy_csv_to_postgres(
    conn,
    "dim_airline",
    [
        "airline_code",
        "airline_key",
        "airline_name"
    ],
    dim_airline_file
)

copy_csv_to_postgres(
    conn,
    "dim_airport",
    [
        "airport_key",
        "airport_code",
        "city_name",
        "state_code",
        "state_name",
        "airport_label"
    ],
    dim_airport_file
)

copy_csv_to_postgres(
    conn,
    "dim_date",
    [
        "date_key",
        "date",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "day_of_month",
        "day_of_week_number",
        "day_name",
        "weekend_flag"
    ],
    dim_date_file
)

print("All dimension tables loaded.")


Loaded dim_airline from dim_airline.csv in 0.0 seconds.
Loaded dim_airport from dim_airport.csv in 0.0 seconds.
Loaded dim_date from dim_date.csv in 0.0 seconds.
All dimension tables loaded.


## 9. Validate Dimension Tables

In [9]:
for table_name in [
    "dim_airline",
    "dim_airport",
    "dim_date"
]:
    cursor.execute(
        f"SELECT COUNT(*) FROM {table_name};"
    )

    row_count = cursor.fetchone()[0]

    print(f"{table_name}: {row_count:,} rows")


dim_airline: 14 rows
dim_airport: 352 rows
dim_date: 365 rows


Expected:

```text
dim_airline : 14
dim_airport : 352
dim_date    : 365
```


## 10. Load 12 Monthly Fact Files

In [10]:
fact_columns = [
    "date_key",
    "airline_key",
    "origin_airport_key",
    "destination_airport_key",
    "flight_number",
    "tail_number",
    "route",
    "crs_dep_time",
    "dep_time",
    "dep_delay",
    "dep_delay_minutes",
    "dep_del15",
    "dep_time_blk",
    "taxi_out",
    "crs_arr_time",
    "arr_time",
    "arr_delay",
    "arr_delay_minutes",
    "arr_del15",
    "arr_time_blk",
    "taxi_in",
    "cancelled",
    "cancellation_code",
    "diverted",
    "crs_elapsed_time",
    "actual_elapsed_time",
    "air_time",
    "distance",
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay",
    "scheduled_dep_hour",
    "departure_period",
    "valid_arrival_flag",
    "arrival_delayed15_flag",
    "on_time_arrival_flag",
    "severe_delay_flag",
    "arrival_status",
    "delay_severity",
    "total_cause_delay_minutes",
    "net_delay_change_minutes",
    "distance_band"
]

print(f"Fact columns: {len(fact_columns)}")


Fact columns: 44


In [13]:
for file_number, file in enumerate(
    fact_files,
    start=1
):
    print("")
    print("=" * 70)
    print(
        f"Loading fact file "
        f"{file_number}/12: {file.name}"
    )

    copy_csv_to_postgres(
        conn,
        "fact_flights",
        fact_columns,
        file,
        compressed=True
    )

    cursor.execute(
        "SELECT COUNT(*) FROM fact_flights;"
    )

    cumulative_rows = cursor.fetchone()[0]

    print(
        f"Cumulative fact rows: "
        f"{cumulative_rows:,}"
    )

print("")
print("All 12 fact files loaded.")



Loading fact file 1/12: fact_flights_2025_01.csv.gz
Loaded fact_flights from fact_flights_2025_01.csv.gz in 52.3 seconds.
Cumulative fact rows: 539,747

Loading fact file 2/12: fact_flights_2025_02.csv.gz
Loaded fact_flights from fact_flights_2025_02.csv.gz in 47.6 seconds.
Cumulative fact rows: 1,044,631

Loading fact file 3/12: fact_flights_2025_03.csv.gz
Loaded fact_flights from fact_flights_2025_03.csv.gz in 67.6 seconds.
Cumulative fact rows: 1,645,503

Loading fact file 4/12: fact_flights_2025_04.csv.gz
Loaded fact_flights from fact_flights_2025_04.csv.gz in 67.6 seconds.
Cumulative fact rows: 2,229,453

Loading fact file 5/12: fact_flights_2025_05.csv.gz
Loaded fact_flights from fact_flights_2025_05.csv.gz in 54.4 seconds.
Cumulative fact rows: 2,835,101

Loading fact file 6/12: fact_flights_2025_06.csv.gz
Loaded fact_flights from fact_flights_2025_06.csv.gz in 53.6 seconds.
Cumulative fact rows: 3,446,676

Loading fact file 7/12: fact_flights_2025_07.csv.gz
Loaded fact_flights

## 11. Final Row Count Validation

In [14]:
expected_fact_rows = 7_001_619

cursor.execute(
    "SELECT COUNT(*) FROM fact_flights;"
)

fact_row_count = cursor.fetchone()[0]

print(f"Expected fact rows: {expected_fact_rows:,}")
print(f"PostgreSQL fact rows: {fact_row_count:,}")

if fact_row_count != expected_fact_rows:
    raise ValueError(
        "Fact row count does not match expected total."
    )

print("Fact row count validation passed.")


Expected fact rows: 7,001,619
PostgreSQL fact rows: 7,001,619
Fact row count validation passed.


## 12. Create Indexes

In [15]:
index_queries = [
    "CREATE INDEX IF NOT EXISTS idx_fact_flights_date_key ON fact_flights(date_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_flights_airline_key ON fact_flights(airline_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_flights_origin_airport ON fact_flights(origin_airport_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_flights_destination_airport ON fact_flights(destination_airport_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_flights_route ON fact_flights(route);"
]

for query in index_queries:
    cursor.execute(query)

conn.commit()

print(f"Indexes created: {len(index_queries)}")


Indexes created: 5


## 13. Analyze Tables

In [16]:
conn.autocommit = True

for table_name in [
    "dim_airline",
    "dim_airport",
    "dim_date",
    "fact_flights"
]:
    cursor.execute(
        f"ANALYZE {table_name};"
    )

print("PostgreSQL statistics updated with ANALYZE.")


PostgreSQL statistics updated with ANALYZE.


## 14. Final Database Validation

In [17]:
validation_queries = {
    "Total flights": "SELECT COUNT(*) FROM fact_flights;",
    "Airlines": "SELECT COUNT(*) FROM dim_airline;",
    "Airports": "SELECT COUNT(*) FROM dim_airport;",
    "Dates": "SELECT COUNT(*) FROM dim_date;",
    "Valid arrivals": '''
        SELECT COUNT(*)
        FROM fact_flights
        WHERE valid_arrival_flag = 1;
    ''',
    "Delayed arrivals 15+": '''
        SELECT COUNT(*)
        FROM fact_flights
        WHERE arrival_delayed15_flag = 1;
    ''',
    "Cancelled flights": '''
        SELECT COUNT(*)
        FROM fact_flights
        WHERE cancelled = 1;
    ''',
    "Diverted flights": '''
        SELECT COUNT(*)
        FROM fact_flights
        WHERE diverted = 1;
    '''
}

print("DATABASE VALIDATION")
print("=" * 70)

for label, query in validation_queries.items():
    cursor.execute(query)
    result = cursor.fetchone()[0]

    print(f"{label}: {result:,}")


DATABASE VALIDATION
Total flights: 7,001,619
Airlines: 14
Airports: 352
Dates: 365
Valid arrivals: 6,879,484
Delayed arrivals 15+: 1,534,638
Cancelled flights: 102,876
Diverted flights: 19,258


## 15. Quick KPI Validation

In [18]:
kpi_query = '''
    SELECT
        COUNT(*) AS total_flights,
        SUM(valid_arrival_flag) AS valid_arrivals,
        SUM(arrival_delayed15_flag) AS delayed_arrivals,
        ROUND(
            100.0
            * SUM(arrival_delayed15_flag)
            / NULLIF(SUM(valid_arrival_flag), 0),
            2
        ) AS arrival_delay_rate_pct,
        ROUND(
            100.0
            * SUM(cancelled)
            / COUNT(*),
            2
        ) AS cancellation_rate_pct,
        ROUND(
            100.0
            * SUM(diverted)
            / COUNT(*),
            2
        ) AS diversion_rate_pct
    FROM fact_flights;
'''

cursor.execute(kpi_query)
kpi_result = cursor.fetchone()

print(f"Total flights: {kpi_result[0]:,}")
print(f"Valid arrivals: {kpi_result[1]:,}")
print(f"Delayed arrivals 15+: {kpi_result[2]:,}")
print(f"Arrival delay rate: {kpi_result[3]}%")
print(f"Cancellation rate: {kpi_result[4]}%")
print(f"Diversion rate: {kpi_result[5]}%")


Total flights: 7,001,619
Valid arrivals: 6,879,484
Delayed arrivals 15+: 1,534,638.0
Arrival delay rate: 22.31%
Cancellation rate: 1.47%
Diversion rate: 0.28%


## 16. Close Connection

In [19]:
cursor.close()
conn.close()

print("PostgreSQL connection closed.")


PostgreSQL connection closed.
